In [ ]:
%%time
import numpy as np
from utilities import states_from_fortran_restarts, setup_infrastructure

from pyshield.radiation import RTE_RRTMGPState, RTE_RRTMGPConfig, RTE_RRTMGPDriver
from pyshield.physics_state import SurfaceState, PhysicsState, PHYSICS_PACKAGES
from pyshield._config import PhysicsConfig
from pyshield.stencils.physics import Physics, flip_field_k

import matplotlib.pyplot as plt
import datetime
from pathlib import Path

In [ ]:
dycore_data = Path("../../test_data/radtest/fv_core.res.tile1.nc")
phys_data = Path("../../test_data/radtest/phy_data.tile1.nc")
tracer_data = Path("../../test_data/radtest/fv_tracer.res.tile1.nc")

In [ ]:
nx = 48
ny = 48
nz = 91
npx = nx + 1
npy = ny + 1
npz = nz + 1
levels = np.arange(npz)
layers = np.arange(nz)

date = datetime.datetime(2020, 1, 1, 12, tzinfo=datetime.timezone.utc)

In [ ]:
quantity_factory, stencil_factory, grid_data = setup_infrastructure(nx, ny, nz, "eta91.nc")
# Quick fix:
grid_data.lon_agrid.field[:] = grid_data.lon.field[:-1,:-1]
grid_data.lat_agrid.field[:] = grid_data.lat.field[:-1,:-1]

In [ ]:
conf = PhysicsConfig(
    dt_atmos=225.0,
    fhswr=225.0,
    fhlwr=225.0,
    npx=npx,
    npy=npy,
    npz=npz,
    nwat=6,
    prescribe_sst=False,
    schemes=["GFS_microphysics", "RTE_RRTMGP"]
)
radconf = RTE_RRTMGPConfig(
    deltsw = 3600.0,
    delt_rad = 3600.0,
    date=date,
    fhswr=1.0,
    fhlwr=1.0,
    isolar=10,
    icmphys=4,
    ico2flg=0,
    ioznflg=1,
    ictmflg=-1,
    ialbflg=-2,
    iemsflg=0,
    ldisable_radiation_quasi_sea_ice=False,
    solar_constant_file=Path("global_solarconstant_noaa_an.txt"),
    input_dir=Path("../../test_data/"),
    aerosol_file=Path("../../test_data/"),
    sollat=0.0,
    nstp=6,
    ivflip=1,
    lcnorm=False,
    lcrick=False,
    gfs_cloud_overlap=False,
)

In [ ]:
state, radstate, sstate = states_from_fortran_restarts(dycore_data, phys_data, tracer_data, grid_data.ak, quantity_factory, conf.schemes)
# state.prsi.field[0, 0, :]

In [ ]:
prsl = (state.prsi.field[:, :, 1:] - state.prsi.field[:, :, :-1]) / np.log(state.prsi.field[:, :, 1:] / state.prsi.field[:, :, :-1])
# prsl[0,0,:]
# state.delp.field[:] = prsl

In [ ]:
%%time
physics = Physics(
    stencil_factory,
    quantity_factory,
    grid_data,
    conf,
    radconf,
    hydro_delp=False,
)

In [ ]:
%%time
physics(state, radstate, sstate, date, conf.dt_atmos)

In [ ]:
# xcol, ycol = np.unravel_index(np.argmax(radstate.qliquid.field.sum(axis=2) + radstate.qice.field.sum(axis=2)), radstate.qliquid.field.sum(axis=2).shape)
xcol, ycol, _ = np.unravel_index(np.argmax(radstate.qliquid.field[:] + radstate.qice.field[:]), radstate.qliquid.field.shape)
print(xcol, ycol)
print(radstate.qliquid.field[xcol,ycol,:].sum())
print(grid_data.lon_agrid.field[xcol,ycol]*180/3.141, grid_data.lat_agrid.field[xcol, ycol]*180/3.141)
print(f"{date} UTC")

In [ ]:
fig0 = plt.figure(0, figsize = (13,13))    #create  figure
f0ax1 = fig0.add_subplot(111) # define axes in which to plot
f0ax1.yaxis.set_inverted(True)
f0ax1.set_xlabel("flux (W/m^2)")
f0ax1.set_ylabel("pressure (hPa)")
f0ax1.plot(radstate.fswd.field[xcol, ycol,:], radstate.prsi.field[xcol, ycol, :]/100, label="SW all-sky flux Down")
f0ax1.plot(radstate.fswu.field[xcol, ycol,:], radstate.prsi.field[xcol, ycol, :]/100, label="SW all-sky flux Up")

f0ax2 = f0ax1.twiny()
f0ax2.set_xlabel("specific_humidity (kg/kg)")
f0ax2.plot(radstate.qvapor.field[xcol, ycol,:], radstate.prsi.field[xcol, ycol,:-1]/100, label="qvapor", color="m")

fig0.legend(frameon=False, loc=4)

In [ ]:
plt.plot(radstate.fswd.view[xcol, ycol,:], levels, label="SW all-sky flux Down")
plt.plot(radstate.fswu.view[xcol, ycol,:], levels, label="SW all-sky flux Up")
plt.plot(radstate.fswd_clr.view[xcol, ycol,:], levels, label="SW clear-sky flux Down")
plt.plot(radstate.fswu_clr.view[xcol, ycol,:], levels, label="SW clear-sky flux Up")
plt.legend(frameon=False)

In [ ]:
plt.plot(radstate.flwd.view[xcol, ycol,:], levels, label="LW all-sky flux Down")
plt.plot(radstate.flwu.view[xcol, ycol,:], levels, label="LW all-sky flux Up")
plt.plot(radstate.flwd_clr.view[xcol, ycol,:], levels, label="LW clear-sky flux Down")
plt.plot(radstate.flwu_clr.view[xcol, ycol,:], levels, label="LW clear-sky flux Up")
plt.legend(frameon=False)